In [3]:
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

@dataclass
class ElectionConfig:
    year: int = 2020
    country: str = "US"
    region_filter: str = "US"
    election_date: str = "2020-11-03"
    ad_period_start: str = "2020-01-01"
    ad_period_end: str = "2020-11-03"
    offices: tuple = ("US PRESIDENT",)
    competition_parties: tuple = ("DEMOCRAT", "REPUBLICAN")
    competition_classes: tuple = (
        "Democratic", "Republican", "Anti-Republican", "Anti-Trump"
    )

CFG = ElectionConfig()

ELECTION_STAGES = pd.DataFrame([
    {"stage": "dem_primary",   "stage_start": "2020-02-03", "stage_end": "2020-06-06", "election_datetime": "2020-06-06"},
    {"stage": "conventions",   "stage_start": "2020-06-07", "stage_end": "2020-08-31", "election_datetime": "2020-08-20"},
    {"stage": "general",       "stage_start": "2020-09-01", "stage_end": "2020-11-03", "election_datetime": "2020-11-03"},
    {"stage": "post_election", "stage_start": "2020-11-04", "stage_end": "2020-12-31", "election_datetime": None},
])
ELECTION_STAGES["stage_start"] = pd.to_datetime(ELECTION_STAGES["stage_start"])
ELECTION_STAGES["stage_end"] = pd.to_datetime(ELECTION_STAGES["stage_end"])

AD_SIDE_MAP = {
    "Democratic":       "democratic",
    "Anti-Republican":  "democratic",
    "Anti-Trump":       "democratic",
    "Republican":       "republican",
}

ELECTION_PARTY_SIDE = {
    "DEMOCRAT":   "democratic",
    "REPUBLICAN": "republican",
}

def load_political_ads_data(data_dir: str = "."):
    """
    Load all political ads datasets from CSV files.
    
    Returns dictionary with all datasets.
    """
    files = {
        "full_data": "./final/political_ads_full_data.csv",
        "weekly_summary": "./final/political_ads_weekly_summary.csv",
        "stage_summary": "./final/political_ads_stage_summary.csv",
        "cumulative_stage": "./final/political_ads_cumulative_stage.csv",
        "competition_summary": "./final/competition_summary.csv",
    }
    
    datasets = {}
    for name, filename in files.items():
        filepath = Path(data_dir) / filename
        if filepath.exists():
            datasets[name] = pd.read_csv(filepath)
            print(f"Loaded {name}: {len(datasets[name]):,} rows")
        else:
            print(f"Warning: {filename} not found")
            datasets[name] = None
    
    return datasets

In [4]:
datasets = load_political_ads_data(".")
    
if datasets["full_data"] is not None:
    print("\n" + "=" * 60)
    print("QUICK EXAMPLES")
    print("=" * 60)
    
    print("\n1. Total spend by party side:")
    spend_by_party = datasets["full_data"].groupby("party_side")["spend_usd"].sum()
    print(spend_by_party)
    
    print("\n2. Spend by election stage:")
    stage_spend = datasets["stage_summary"].groupby("election_stage")["stage_spend_usd"].sum()
    print(stage_spend)
    
    print("\n3. Top 5 advertisers by spend:")
    top_ads = datasets["full_data"].groupby("advertiser_name")["spend_usd"].sum().nlargest(5)
    for name, spend in top_ads.items():
        print(f"   {name[:50]}: ${spend:,.0f}")
    
    print("\n4. Weekly spending trend (first 10 weeks):")
    weekly_trend = datasets["weekly_summary"].groupby("week_start")["total_spend_usd"].sum().head(10)
    for date, spend in weekly_trend.items():
        print(f"   {date}: ${spend:,.0f}")

/var/folders/z2/kxjb41213z74h6nbpr1f79sr0000gn/T/ipykernel_32704/1305895316.py:61: DtypeWarning: Columns (3,25,31,36,37,38,39,41,42,44,45,46,47,48,49,50,51,95,96) have mixed types. Specify dtype option on import or set low_memory=False.
  datasets[name] = pd.read_csv(filepath)


Loaded full_data: 236,925 rows
Loaded weekly_summary: 2,390 rows
Loaded stage_summary: 30 rows
Loaded cumulative_stage: 32 rows
Loaded competition_summary: 2 rows

QUICK EXAMPLES

1. Total spend by party side:
party_side
Democratic    1.101085e+09
Republican    7.703058e+08
Name: spend_usd, dtype: float64

2. Spend by election stage:
election_stage
Convention          168460050.0
Election Day          2830200.0
General Campaign    587224800.0
Primary             193632500.0
Name: stage_spend_usd, dtype: float64

3. Top 5 advertisers by spend:
   BIDEN FOR PRESIDENT: $258,876,450
   DONALD J. TRUMP FOR PRESIDENT, INC.: $239,717,250
   MIKE BLOOMBERG 2020 INC: $151,090,300
   TRUMP MAKE AMERICA GREAT AGAIN COMMITTEE: $111,757,600
   FF PAC: $62,986,900

4. Weekly spending trend (first 10 weeks):
   2020-01-01: $3,153,750
   2020-01-02: $1,966,350
   2020-01-03: $730,850
   2020-01-04: $240,750
   2020-01-05: $3,267,350
   2020-01-06: $234,950
   2020-01-07: $997,300
   2020-01-08: $66,50